# 1.1 - Data Wrangling

The purpose of this notebook is to transform the data to obtain training labels from the `outcome` column of the data obtained from LL2 API, GCAT, and Course-provided launch data.

In [35]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
from pathlib import Path

In [3]:
csv_path = Path.cwd().parent / 'data' / 'interim' / 'api-launch-data-table.csv'
launch_df = pd.read_csv(csv_path)

In [4]:
launch_df.head(10)

,launch_designator,booster_version,orbit,launch_site,flights,reused,landing_pad,block,serial,longitude,latitude,reused_count,outcome,OrbPay,launch_date,gridfins,legs
0,2020-084,Falcon 9,Low Earth Orbit,Launch Complex 39A,23,False,JRTI,Block 5,B1061,-80.604282,28.608227,0,True ASDS,13.000,2020-11-16,True,True
1,2020-078,Falcon 9,Medium Earth Orbit,Space Launch Complex 40,23,False,OCISLY,Block 5,B1062,-80.577357,28.561941,0,True ASDS,4.400,2020-11-05,True,True
2,2020-074,Falcon 9,Low Earth Orbit,Space Launch Complex 40,20,True,JRTI,Block 5,B1060,-80.577357,28.561941,2,True ASDS,16.000,2020-10-24,True,True
3,2020-073,Falcon 9,Low Earth Orbit,Launch Complex 39A,14,True,OCISLY,Block 5,B1051,-80.604282,28.608227,5,True ASDS,16.000,2020-10-18,True,True
4,2020-070,Falcon 9,Low Earth Orbit,Launch Complex 39A,19,True,OCISLY,Block 5,B1058,-80.604282,28.608227,2,True ASDS,16.000,2020-10-06,True,True
5,2020-062,Falcon 9,Low Earth Orbit,Launch Complex 39A,20,True,OCISLY,Block 5,B1060,-80.604282,28.608227,1,True ASDS,16.000,2020-09-03,True,True
6,2020-059,Falcon 9,Sun-Synchronous Orbit,Space Launch Complex 40,6,True,LZ-1,Block 5,B1059,-80.577357,28.561941,3,True RTLS,3.091,2020-08-30,True,True
7,2020-057,Falcon 9,Low Earth Orbit,Space Launch Complex 40,11,True,OCISLY,Block 5,B1049,-80.577357,28.561941,5,True ASDS,15.815,2020-08-18,True,True
8,2020-055,Falcon 9,Low Earth Orbit,Launch Complex 39A,14,True,OCISLY,Block 5,B1051,-80.604282,28.608227,4,True ASDS,15.335,2020-08-07,True,True
9,2020-048,Falcon 9,Geostationary Transfer Orbit,Space Launch Complex 40,19,True,JRTI,Block 5,B1058,-80.577357,28.561941,1,True ASDS,5.600,2020-07-20,True,True


identify and calculate the percentage of missing values from each column.

In [22]:
print('{}'.format(np.round(100*launch_df[~launch_df.isnull()].count()/launch_df.shape[0],2)))

launch_designator    100.00
booster_version      100.00
orbit                 98.96
launch_site          100.00
flights              100.00
reused               100.00
landing_pad          100.00
block                100.00
serial               100.00
longitude            100.00
latitude             100.00
reused_count         100.00
outcome              100.00
OrbPay               100.00
launch_date          100.00
gridfins             100.00
legs                 100.00
dtype: float64


Identify which columns are numerical or categorical

In [23]:
launch_df.dtypes

launch_designator        str
booster_version          str
orbit                    str
launch_site              str
flights                int64
reused                  bool
landing_pad              str
block                    str
serial                   str
longitude            float64
latitude             float64
reused_count           int64
outcome                  str
OrbPay               float64
launch_date              str
gridfins                bool
legs                    bool
dtype: object

Calculate the number of launches on each site

In [24]:
launch_df['launch_site'].value_counts()

launch_site
Space Launch Complex 40    58
Launch Complex 39A         23
Space Launch Complex 4E    15
Name: count, dtype: int64

Calculate the number and occurences of each orbit

In [25]:
launch_df['orbit'].value_counts()

orbit
Low Earth Orbit                 54
Geostationary Transfer Orbit    28
Sun-Synchronous Orbit            6
Medium Earth Orbit               3
Lunar Orbit                      1
Polar Orbit                      1
High Earth Orbit                 1
Heliocentric L1                  1
Name: count, dtype: int64

Calculate the number and occurence of mission outcome

In [27]:
landing_outcomes = launch_df['outcome'].value_counts()
print(landing_outcomes)

outcome
True ASDS      44
None EXP       20
True RTLS      14
False ASDS      7
None Ocean      6
False Ocean     2
False PCL       2
False RTLS      1
Name: count, dtype: int64


Create a set of bad outcomes -- landing outcomes with False or None.

In [31]:
bad_outcomes = set()
for outcome in landing_outcomes.keys():
    if 'False' in outcome or 'None' in outcome:
        bad_outcomes.add(outcome)

In [32]:
bad_outcomes

{'False ASDS',
 'False Ocean',
 'False PCL',
 'False RTLS',
 'None EXP',
 'None Ocean'}

Create a landing outcome label from `outcome` column.

Here a script is created to create the `Class` column (the target variable for analysis) to the launch data from the specified csv path to the launches DataFrame.

In [37]:
def add_class(launch_csv: str|Path, save_path: None|str|Path = None) -> None|Path:
    """
    Function that adds the target 'Class' column to the launch data from the specified launch csv path,
    and saves the result to the specified save path.

    :param launch_csv: str or Path; file path to the launches csv file
    :param save_path: str or Path; Default None; file path where the result is to be saved. If None,
                      result will be saved in the current working directory.
    :return save_path: Path; file path where result is saved. 
    """
    try:
        launch_df = pd.read_csv(launch_csv)
        
    except Exception as e:
        print(e)
        return None
    
    landing_outcomes = launch_df['outcome'].unique()

    bad_outcomes = set()
    for outcome in landing_outcomes:
        if 'False' in outcome or 'None' in outcome:
            bad_outcomes.add(outcome)
            
    def outcome_map(entry: str) -> int:
        """
        Utility function that maps an outcome to either the integer 1 or 0.
        If the outcome is a landing success, the function returns 1; otherwise,
        the function returns 0

        :param entry: str; a landing outcome in the format "(Landing Success) (Landing Pad)", e.g., True ASDS.
        :return outcome: int; if the entry is a landing success, `outcome=1`; otherwise, `outcome=0`.
        """

        if entry in bad_outcomes:
            outcome = 0
        else:
            outcome = 1

        return outcome

    launch_df['class'] = launch_df['outcome'].map(outcome_map)

    if not save_path:
        save_path = Path.cwd() / 'api-launch-data-table-class.csv'

    if isinstance(save_path, str):
        save_path = Path(save_path)
        
    launch_df.to_csv(save_path, index=False)

    return save_path

In [38]:
# Test the script
from spacey_falcon_9_project.dataset import add_class

launch_csv = Path.cwd().parent / 'data' / 'interim' / 'api-launch-data-table.csv'
save_path = Path.cwd().parent / 'data' / 'interim' / 'api-launch-data-table-class.csv'

add_class(launch_csv, save_path)

PosixPath('/Users/jelo/spacey_falcon_9_project/data/interim/api-launch-data-table-class.csv')

In [43]:
launches_with_class = pd.read_csv(save_path)
launches_with_class[['outcome', 'class']]

,outcome,class
0,True ASDS,1
1,True ASDS,1
2,True ASDS,1
3,True ASDS,1
4,True ASDS,1
...,...,...
91,None EXP,0
92,None EXP,0
93,None EXP,0
94,False PCL,0
